[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/35_bpe.ipynb)

# 🔴 Hard: Byte-Pair Encoding (BPE)

Implement a simple **BPE tokenizer** — the foundation of GPT/LLaMA tokenization.

### Signature
```python
class SimpleBPE:
    def __init__(self): ...
    def train(self, corpus: list[str], num_merges: int): ...
    def encode(self, text: str) -> list[str]: ...
```

### Algorithm (training)
1. Split each word into characters + `</w>` end marker
2. Count all adjacent pairs across the corpus
3. Merge the most frequent pair into a single token
4. Repeat for `num_merges` iterations

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00


In [ ]:
# No imports needed

In [15]:
# ✏️ YOUR IMPLEMENTATION HERE

class SimpleBPE:
    def __init__(self):
        self.merges = []
        self.vocab=set()


    def train(self, corpus, num_merges):
        word_freqs={}
        for word in corpus:
          tokens=tuple(list(word)+["</w>"])
          word_freqs[tokens]=word_freqs.get(tokens,0)+1
        for i in range(num_merges):
          pairs={}
          for tokens , freq in word_freqs.items():
            for j in range (len(tokens)-1):
              pair=(tokens[j],tokens[j+1])
              pairs[pair]=pairs.get(pair,0)+1
          if not pairs:
            break

          best_pair=None
          max_freq = -1
          for pair,freq in pairs.items():
            if freq > max_freq:
              max_freq=freq

              best_pair=pair

          if best_pair==None:
            break

          self.merges.append(best_pair)


          new_word_freqs={}
          first, second = best_pair
          new_token = first + second

          for token , freq in word_freqs.items():
            new_tokens=[]
            j=0
            while j < len(tokens):
              if j < len(tokens)-1 and tokens[j]==first and tokens[j+1]==second:
                new_tokens.append(new_token)
                j+=2
              else:
                new_tokens.append(tokens[j])
                j+=1
            new_word_freqs[tuple(new_tokens)]=freq
          word_freqs=new_word_freqs




    def encode(self, text):
        words=text.split()
        final_tokens=[]
        for word in words:
          tokens=list(word)+["</w>"]

          for (first,second) in self.merges:
            new_token=first+second
            new_list=[]
            i=0
            while i < len(tokens):
              if i < len(tokens)-1 and tokens[i]==first and tokens[i+1] == second:
                i+=2
                new_list.append(first+second)
              else:

                new_list.append(tokens[i])
                i+=1

            tokens=new_list
          final_tokens.extend(tokens)
        return final_tokens

In [16]:
# 🧪 Debug
bpe = SimpleBPE()
bpe.train(['low', 'low', 'low', 'lower', 'newest', 'widest'], num_merges=10)
print('Merges:', bpe.merges[:5])
print('Encode:', bpe.encode('low lower'))

Merges: [('l', 'o'), ('w', 'i'), ('wi', 'd'), ('wid', 'e'), ('wide', 's')]
Encode: ['lo', 'w', '</w>', 'lo', 'w', 'e', 'r', '</w>']


In [17]:
# ✅ SUBMIT
from torch_judge import check
check('bpe')


🧪 Testing: Byte-Pair Encoding (BPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Correct number of merges (0.3ms)
  ✅ [2/4] Most frequent pair merged first (0.1ms)
  ✅ [3/4] Encode returns list of strings (0.3ms)
  ✅ [4/4] More merges -> fewer tokens (0.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (0.9ms total)
  Progress saved. Run status() to see your dashboard.

